# **TUTORIAL** 4: Run the time-explicit prospective LCA

Time-explicit prospective life cycle assessment (LCA) quantifies the environmental
burdens associated with the annual material flows calculated by the MFA.
`run_lca(...)` links each foreground activity to the background inventory
corresponding to its year of occurrence and reports environmental burdens over
2019--2100.

## Before starting...

Complete [Tutorial 2](2_prepare_backgrounds.ipynb) and
[Tutorial 3](3_run_mfa.ipynb) for the backgrounds and foreground scenario you
want to assess.


In [ ]:
import json
from pathlib import Path

import pandas as pd

from pytoce import run_lca, set_workspace

## Select the workspace

Run `set_workspace(...)` before any later function call. It sets the active
workspace root for the current Python session. Use the directory from Tutorial 1.


In [ ]:
workspace = Path.home() / "Documents" / "pytoce_workspace"
set_workspace(workspace)

## Calculate the environmental burdens

This example calculates the environmental burdens of *2019frozen_SSP2demand*
using `frozen_2019_default_baseline`. These results support monitoring and
provide the baseline LCA for the target-setting example.

The call uses the four REMIND SSP2 background scenarios described in
[Tutorial 2](2_prepare_backgrounds.ipynb), the 16 EF 3.1 midpoint impact
categories, and the ten supported PB-LCIA impact categories. Detailed upstream
contribution analysis is calculated for 2019, 2050, and 2100. Each foreground
and background scenario pair has its own result. Later calls reuse completed
results.


In [ ]:
foreground = "frozen_2019_default_baseline"
lca_report = run_lca(foreground_scenario=foreground)
lca_report.as_dict()

## Read annual environmental burdens

`annual_scores.parquet` reports regional and global environmental burdens for
each year from 2019 to 2100. `scores.parquet` sums them over 2019--2100.
`annual_life_cycle_stage_scores.parquet` and `life_cycle_stage_scores.parquet`
separate the annual and period-total environmental burdens by life cycle stage.

The following cell selects the global EF 3.1 climate change result in 2050 for
each background. `global_derived` is the sum of the 12 regional scores for the
same LCIA method, impact category, and year or period. The `unit` column gives
the unit of each impact category.

Production, conversion, recycling, and end-of-life treatment are evaluated with
the background system corresponding to their year of occurrence. Emissions are
assigned to the year of the corresponding activity, including emissions spread
over several years within a treatment inventory. Characterization factors are
kept constant across years.


In [ ]:
climate_2050 = pd.concat(
    [
        pd.read_parquet(
            result.annual_scores,
            filters=[
                ("method_id", "==", "ef31"),
                ("impact_category_id", "==", "climate_change"),
                ("year", "==", 2050),
                ("region", "==", "global_derived"),
            ],
        ).assign(background_scenario=result.background_scenario)
        for result in lca_report.results
        if result.foreground_scenario == "frozen_2019_default_baseline"
    ],
    ignore_index=True,
)
climate_2050[["background_scenario", "year", "impact_category_id", "unit", "score"]]

`contributions.parquet` reports detailed contributions for the years selected
with `upstream_analysis_year`; `stage_contributions.parquet` separates them by
life cycle stage. Annual and period-total scores cover the full 2019--2100 period.

`modelled_direct` identifies contributions from pyToCE's process models for
polyamide fibers (PA6 and PA66), including their wastewater and waste treatment.
Each row names the process in `receiving_function` and the environmental flow,
such as a CO2 emission, in `elementary_flow_name`. Its `score` is included in
the reported total for the same region, year, and impact category.

`contribution_norm` is the sum of the absolute values of the characterized
contributions, in the impact category's unit. Regional results follow the MFA
regions, while `activity_location` records the upstream activity geography
available in the selected inventories.


## Record the calculation details

The manifest identifies the MFA result, prepared background, LCIA methods, software versions, and result files. Keep it with the tables when a result is cited or shared.

In [ ]:
lca_result = lca_report.results[0]
manifest = json.loads(lca_result.manifest_path.read_text(encoding="utf-8"))
{
    "result_identity": manifest["identity"],
    "foreground_scenario": manifest["foreground_scenario"],
    "background_scenario": manifest["background_scenario"],
    "manifest_path": lca_result.manifest_path,
}

## What to do next

Calculate allocated carrying capacities following the
[optional pyaesa tutorial](../optional_workflows/pyaesa_allocated_carrying_capacity.ipynb),
or supply your own values based on your chosen allocation method and shares.
The pyaesa tutorial uses the 2019 LCA results calculated here.

Then choose the notebook for your study objective:

- **Monitoring:** follow [Tutorial 5m](5m_run_asr.ipynb) to call `run_asr(...)`
  and compare the environmental burdens of user-defined CE performance
  trajectories with allocated carrying capacities.
- **Target setting:** follow [Tutorial 5o](5o_run_optimization.ipynb) to call
  `run_optimization(...)` and search for CE performance target levels.
